In [49]:
import pandas as pd

df = pd.read_csv("data/StudentPerformanceFactors.csv")

# Remover linhas com qualquer valor vazio para manter a integridade
df = df.dropna().copy()

df.head()
# print(f"Dados prontos. Total de alunos para análise: {len(df)}")

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [50]:
# Risco de Presença: <70 (2 pts), <85 (1 pt), else 0
def categorizar_presenca(valor):
    if valor < 70: return 2
    if valor < 85: return 1
    return 0
df['Attendance_Risk'] = df['Attendance'].apply(categorizar_presenca)

# Risco de Tendência: Caiu nota e está abaixo de 80 (1 pt)
def calcular_tendencia(linha):
    if linha['Exam_Score'] <= linha['Previous_Scores'] and linha['Exam_Score'] < 80:
        return 1
    return 0
df['Trend_Risk'] = df.apply(calcular_tendencia, axis=1)

# Tutoring: 0 sessões (1 pt de risco), >0 sessões (0 pts)
df['Tutoring_Risk'] = df['Tutoring_Sessions'].apply(lambda x: 1 if x == 0 else 0)

# Mapeamentos Inversos (Low=2, Medium=1, High=0)
mapa_inv = {'Low': 2, 'Medium': 1, 'High': 0}
df['Parental_Risk'] = df['Parental_Involvement'].map(mapa_inv)
df['Resources_Risk'] = df['Access_to_Resources'].map(mapa_inv)
df['Motivation_Risk'] = df['Motivation_Level'].map(mapa_inv)
df['Teacher_Risk'] = df['Teacher_Quality'].map(mapa_inv)

# Outros Fatores
df['Disability_Risk'] = df['Learning_Disabilities'].map({'Yes': 1, 'No': 0})
df['Distance_Risk'] = df['Distance_from_Home'].map({'Near': 0, 'Moderate': 1, 'Far': 2})

# --- FATOR DE PROTEÇÃO (SUBTRAI PONTOS) ---
# Atividades Extras: Sim (-1 pt), Não (0 pt)
df['Extracurricular_Bonus'] = df['Extracurricular_Activities'].map({'Yes': -1, 'No': 0})

# --- CÁLCULO FINAL ---

# Soma máxima teórica de riscos = 15 pontos
# (Parental 2 + Resources 2 + Motivation 2 + Attendance 2 + Trend 1 + Teacher 2 + Disability 1 + Distance 2 + Tutoring 1)
max_pontos = 15

df['Vulnerability_Points'] = (
    df['Parental_Risk'] + df['Resources_Risk'] + df['Motivation_Risk'] + 
    df['Attendance_Risk'] + df['Trend_Risk'] + df['Teacher_Risk'] + 
    df['Disability_Risk'] + df['Distance_Risk'] + df['Tutoring_Risk'] + 
    df['Extracurricular_Bonus']
)

# Ajuste: O índice não pode ser negativo e nem ultrapassar o máximo
df['Vulnerability_Points'] = df['Vulnerability_Points'].clip(lower=0, upper=max_pontos)

# Probabilidade 0-100%
df['Probability_of_Issue'] = (df['Vulnerability_Points'] / max_pontos) * 100
df['Probability_of_Issue'] = df['Probability_of_Issue'].round(2)

# Categorização de Risco
def definir_categoria(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    if prob <= 80: return 'Alto'
    return 'Crítico'

df['Risk_Level'] = df['Probability_of_Issue'].apply(definir_categoria)
colunas_visao = ['Teacher_Quality', 'Tutoring_Sessions', 'Extracurricular_Activities', 'Previous_Scores', 'Exam_Score', 'Parental_Involvement', 'Motivation_Level', 'Attendance', 'Probability_of_Issue']

# Visualização do Top 10 Alunos em Risco
df[colunas_visao].sort_values(by='Probability_of_Issue', ascending=False).head(10)

,Teacher_Quality,Tutoring_Sessions,Extracurricular_Activities,Previous_Scores,Exam_Score,Parental_Involvement,Motivation_Level,Attendance,Probability_of_Issue
368,Medium,0,No,100,63,Medium,Low,68,80.00
3543,Medium,0,Yes,66,57,Low,Low,67,80.00
2893,Low,0,Yes,71,59,Medium,Low,60,80.00
1042,Low,0,No,67,65,Low,Low,88,80.00
5362,Low,0,No,64,61,Low,Low,70,80.00
5168,Medium,0,Yes,84,59,Low,Low,64,80.00
1245,Low,0,Yes,66,59,Low,Medium,68,73.33
283,Medium,0,Yes,68,59,Medium,Low,69,73.33
3263,Low,0,No,54,61,Low,Medium,67,73.33
1700,Medium,0,No,98,64,Low,Low,69,73.33


In [51]:
# A soma máxima possível de todos os riscos mapeados é 15
max_pontos = 15

# Somamos todos os riscos e aplicamos o bônus negativo
df['Vulnerability_Points'] = (
    df['Parental_Risk'] + df['Resources_Risk'] + df['Motivation_Risk'] + 
    df['Attendance_Risk'] + df['Trend_Risk'] + df['Teacher_Risk'] + 
    df['Disability_Risk'] + df['Distance_Risk'] + df['Tutoring_Risk'] + 
    df['Extracurricular_Bonus']
)

# Travamos o valor entre 0 e 15 (para não ter risco negativo ou maior que 100%)
df['Vulnerability_Points'] = df['Vulnerability_Points'].clip(lower=0, upper=max_pontos)

# Transformamos a pontuação bruta em uma probabilidade de 0 a 100%
df['Probability_of_Issue'] = (df['Vulnerability_Points'] / max_pontos) * 100
df['Probability_of_Issue'] = df['Probability_of_Issue'].round(2)

In [59]:
# Criamos as faixas de risco para facilitar a filtragem no dashboard
def definir_categoria(prob):
    if prob <= 20: return 'Baixo'
    if prob <= 50: return 'Médio'
    if prob <= 80: return 'Alto'
    return 'Crítico'

df['Risk_Level'] = df['Probability_of_Issue'].apply(definir_categoria)

# Selecionamos as colunas principais para a sua análise final
colunas_visao = [
    'Teacher_Quality', 'Tutoring_Sessions', 'Extracurricular_Activities', 
    'Previous_Scores', 'Exam_Score', 'Parental_Involvement', 
    'Motivation_Level', 'Attendance', 'Probability_of_Issue', 'Risk_Level'
]

# 1. Organizamos a lista completa de comparação (Original vs Risco)
# Cada linha aqui representa um dos fatores que somam os 15 pontos
colunas_full_detalhe = [
    # Fatores Qualitativos (0 a 2 pts)
    'Parental_Involvement', 'Parental_Risk',
    'Access_to_Resources', 'Resources_Risk',
    'Motivation_Level', 'Motivation_Risk',
    'Teacher_Quality', 'Teacher_Risk',
    'Distance_from_Home', 'Distance_Risk',
    
    # Fatores de Frequência e Histórico (0 a 2 pts / 0 a 1 pt)
    'Attendance', 'Attendance_Risk',
    'Previous_Scores', 'Exam_Score', 'Trend_Risk',
    
    # Fatores Binários e Saúde (0 a 1 pt)
    'Tutoring_Sessions', 'Tutoring_Risk',
    'Learning_Disabilities', 'Disability_Risk',
    
    # Fator de Proteção (Bônus -1 pt)
    'Extracurricular_Activities', 'Extracurricular_Bonus',
    
    # Resultados Finais
    'Probability_of_Issue', 'Risk_Level'
]

# 2. Ordenamos pelo maior risco
df_ordenado = df.sort_values(by='Probability_of_Issue', ascending=False)

# 3. Exibimos os 3 primeiros com todas as colunas de rastro
df_full_3 = df_ordenado[colunas_full_detalhe].head(3)

print("--- AUDITORIA COMPLETA DE DADOS: VALOR BRUTO VS PESO DE RISCO ---")
# Usamos o .T (transpor) para facilitar a leitura, 
# assim cada aluno vira uma coluna e você vê as regras linha a linha
display(df_full_3.T)

df_ordenado = df.sort_values(by='Probability_of_Issue', ascending=True)
df_full_3 = df_ordenado[colunas_full_detalhe].head(3)
display(df_full_3.T)

--- AUDITORIA COMPLETA DE DADOS: VALOR BRUTO VS PESO DE RISCO ---


,368,3543,2893
Parental_Involvement,Medium,Low,Medium
Parental_Risk,1,2,1
Access_to_Resources,Low,Low,Medium
Resources_Risk,2,2,1
Motivation_Level,Low,Low,Low
Motivation_Risk,2,2,2
Teacher_Quality,Medium,Medium,Low
Teacher_Risk,1,1,2
Distance_from_Home,Far,Far,Far
Distance_Risk,2,2,2


,1682,5730,5549
Parental_Involvement,High,High,Medium
Parental_Risk,0,0,1
Access_to_Resources,High,High,High
Resources_Risk,0,0,0
Motivation_Level,Medium,High,High
Motivation_Risk,1,0,0
Teacher_Quality,High,High,High
Teacher_Risk,0,0,0
Distance_from_Home,Near,Near,Near
Distance_Risk,0,0,0


In [ ]:
def get_student_diagnostic(aluno_row):
    causas = []
    
    # Verificando cada fator para "explicar" o risco
    if aluno_row['Attendance_Risk'] == 2:
        causas.append("baixa frequência escolar (crítica)")
    if aluno_row['Motivation_Risk'] == 2:
        causas.append("baixa motivação intrínseca")
    if aluno_row['Teacher_Risk'] == 2:
        causas.append("percepção de baixa qualidade do corpo docente")
    if aluno_row['Tutoring_Risk'] == 1:
        causas.append("falta de acompanhamento em tutorias/reforço")
    if aluno_row['Trend_Risk'] == 1:
        causas.append("queda recente no desempenho acadêmico")
    if aluno_row['Parental_Risk'] == 2:
        causas.append("baixo envolvimento dos responsáveis")

    # Construindo a frase
    if not causas:
        return "Aluno apresenta perfil estável com baixo índice de vulnerabilidade."
    
    frase_inicial = f"O aluno possui {aluno_row['Probability_of_Issue']}% de risco. "
    detalhes = "Fatores principais: " + ", ".join(causas) + "."
    
    return frase_inicial + detalhes

# Testando para o primeiro aluno da lista de risco
aluno_exemplo = df.sort_values(by='Probability_of_Issue', ascending=False).iloc[0]
print(get_student_diagnostic(aluno_exemplo))

O aluno possui 80.0% de risco. Fatores principais: baixa frequência escolar (crítica), baixa motivação intrínseca, falta de acompanhamento em tutorias/reforço, queda recente no desempenho acadêmico.
